In [1]:
!pip install -q transformers sentence-transformers torch

In [2]:
#Extracting and Inspecting Attention Weights
#We can instruct Hugging Face models to output their raw attention matrices so we can see which tokens "pay attention" to which.
import torch
from transformers import AutoModel, AutoTokenizer

# Load a lightweight BERT or GPT model with attention outputs enabled
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_attentions=True)

# Define sample text
text = "The bank of the river was muddy."
inputs = tokenizer(text, return_tensors="pt")

# Pass through the model
with torch.no_grad():
    outputs = model(**inputs)

# Extract attention weights across all layers
# Shape of attentions: tuple of (batch_size, num_heads, sequence_length, sequence_length)
attentions = outputs.attentions

print("Number of Layers:", len(attentions))
print("Attention Tensor Shape per Layer:", attentions[0].shape)
# Output shape: [1, 12, 9, 9] -> 1 batch, 12 heads, 9 tokens x 9 tokens

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of Layers: 12
Attention Tensor Shape per Layer: torch.Size([1, 12, 10, 10])


In [3]:
#Visualizing Attention Patterns with bertviz
#To make attention visible, we can use the bertviz library in Colab to interactively inspect how tokens attend to each other layer-by-layer and head-by-head.
# Install bertviz if not already installed
!pip install -q bertviz

from bertviz import head_view
from transformers import AutoTokenizer, AutoModel

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_attentions=True)

inputs = tokenizer("The bank of the river was muddy.", return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

with torch.no_grad():
    outputs = model(**inputs)

# Render interactive attention head view in Colab
head_view(outputs.attentions, tokens)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 10.4 MB/s eta 0:00:00


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<IPython.core.display.Javascript object>

In [4]:
#Observing Contextual Vector Evolution Across Layers
#Observe how token embeddings change as they pass from the first Transformer layer to the deep final layer.
# Re-run model with hidden states enabled
model = AutoModel.from_pretrained(model_name, output_hidden_states=True)

with torch.no_grad():
    outputs = model(**inputs)

# outputs.hidden_states contains vectors for inputs + output of each layer
hidden_states = outputs.hidden_states

# Extract token 'bank' (index 2) vector at Layer 0 (input) vs Layer 12 (final)
initial_vector = hidden_states[0][0, 2]
final_vector = hidden_states[-1][0, 2]

# Measure cosine similarity between initial static representation and final contextualized representation
cosine_sim = torch.nn.functional.cosine_similarity(initial_vector, final_vector, dim=0)

print("Layer 0 (Initial) Vector Shape:", initial_vector.shape)
print("Layer 12 (Final) Vector Shape:", final_vector.shape)
print(f"Similarity between raw & fully contextualized 'bank' vector: {cosine_sim.item():.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Layer 0 (Initial) Vector Shape: torch.Size([768])
Layer 12 (Final) Vector Shape: torch.Size([768])
Similarity between raw & fully contextualized 'bank' vector: 0.2425
